# Homework 6 — State-Space Models and Kalman Filtering

**Coverage:** Lectures 19–20<br>
**Due:** Sunday, October 25, 2026, 11:59 p.m. ET<br>
**Total:** 100 points

## Instructions

- Complete this notebook in Google Colab.
- Problem 1 is a manual mathematics problem. Show every important
  step in Markdown/LaTeX, or insert one clearly legible image of
  your handwritten derivation. Code may check arithmetic only
  after the derivation is complete.
- Problem 2 is a scaffolded scientific-computing study. Use the
  supplied random seeds and do not delete setup, helper, or check
  cells.
- Your submitted notebook must run from beginning to end in a
  fresh Colab runtime without Google Drive, absolute paths, or
  additional package installation.
- Label plots and include documented units. If a legacy dataset
  has no documented units, label the quantity as normalized or
  unit-unspecified rather than inventing units. Unless stated
  otherwise, report numerical answers to at least four
  significant digits.

## Student details

- **First name:**
- **Last name:**
- **Purdue email:**


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import expm

SEED = 53906
rng = np.random.default_rng(SEED)
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(precision=6, suppress=True)

# During drafting this points to master. Before release, the instructor
# will replace DATA_REVISION with the immutable course release tag.
DATA_REVISION = "master"
DATA_BASE = (
    "https://raw.githubusercontent.com/PredictiveScienceLab/"
    f"data-analytics-se/{DATA_REVISION}/lecturebook/data/homework"
)

def course_data(name):
    local_candidates = [
        Path("../data/homework") / name,
        Path("lecturebook/data/homework") / name,
    ]
    for local in local_candidates:
        if local.exists():
            return local
    return f"{DATA_BASE}/{name}"


## Problem 1 — One Kalman-filter step (25 points)

For \(\ddot q+2\zeta\omega_0\dot q+\omega_0^2q=u(t)\), let
\(x=(q,\dot q)^T\), \(\omega_0=2\), \(\zeta=0.1\), and
\(\Delta t=0.1\). Use forward Euler and

\[
m_0=(0.5,-0.2)^T,\quad
P_0=\begin{bmatrix}.04&.01\\.01&.09\end{bmatrix},\quad u_0=.3,
\]
\[
Q=\operatorname{diag}(.001,.004),\quad H=[1\;0],\quad
R=.01,\quad y_1=.42.
\]

1. Derive the continuous matrix, \(F=I+\Delta tA_c\), and \(B\). **(5)**
2. Calculate the predicted mean and covariance. **(6)**
3. Calculate innovation, innovation variance, and gain. **(6)**
4. Calculate the updated mean and Joseph-form covariance
   \(P_1=(I-KH)P_1^-(I-KH)^T+KRK^T\); verify symmetry. **(6)**
5. Describe the \(R\to0\) limit and why exact position does not
   necessarily imply zero velocity variance. **(2)**


> **Response:** Replace this text with your work.


## Problem 2 — Filtering a hidden forced oscillator (75 points)

The cell below creates a reproducible trajectory and noisy position
observations with separate process and measurement random streams.


In [ ]:
omega0, zeta = 2.0, 0.15
dt, final_time = 0.05, 20.0
force_amplitude, force_frequency = 0.6, 1.2
Ac = np.array([[0.0, 1.0], [-omega0**2, -2*zeta*omega0]])
b = np.array([[0.0], [1.0]])
aug = np.zeros((3, 3)); aug[:2, :2] = Ac; aug[:2, 2:] = b
disc = expm(aug * dt)
F, B = disc[:2, :2], disc[:2, 2]
Q_TRUE = np.diag([2e-5, 5e-4])
H = np.array([[1.0, 0.0]])
R = np.array([[0.04**2]])
FILTER_M0 = np.array([0.0, 0.0])
FILTER_P0 = np.diag([0.25, 0.25])
t = np.arange(0.0, final_time + dt/2, dt)
u = force_amplitude * np.cos(force_frequency * t[:-1])
process_rng = np.random.default_rng(5390601)
measurement_rng = np.random.default_rng(5390602)
x_true = np.empty((len(t), 2)); x_true[0] = [0.35, -0.20]
for k in range(len(t) - 1):
    x_true[k+1] = (
        F @ x_true[k]
        + B*u[k]
        + process_rng.multivariate_normal(np.zeros(2), Q_TRUE)
    )
y = (H @ x_true.T).ravel() + measurement_rng.normal(
    0.0, np.sqrt(R[0, 0]), len(t)
)
print("Steps:", len(t), "F[0,:] =", F[0])


In [ ]:
def kalman_filter(y, u, F, B, H, Q, R, m0, P0):
    '''Filter y with a fixed update/predict convention.

    First assimilate y[0] into the prior (m0, P0) at t[0] and
    store that posterior at index 0. For k=0,...,len(u)-1,
    predict with u[k] to t[k+1], assimilate y[k+1], and store the
    posterior at index k+1.

    Return means (n,2), covariances (n,2,2), innovations (n,),
    and innovation variances (n,). Use a linear solve for the gain
    and the Joseph update
    P = (I-KH) @ P_minus @ (I-KH).T + K @ R @ K.T.
    '''
    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError

def validate_filter_outputs(means, covariances, innovations,
                            innovation_variances, n_steps):
    '''Run deterministic structural checks on a filter result.'''
    means = np.asarray(means, float)
    covariances = np.asarray(covariances, float)
    innovations = np.asarray(innovations, float)
    innovation_variances = np.asarray(innovation_variances, float)
    assert means.shape == (n_steps, 2)
    assert covariances.shape == (n_steps, 2, 2)
    assert innovations.shape == (n_steps,)
    assert innovation_variances.shape == (n_steps,)
    assert all(np.all(np.isfinite(a)) for a in (
        means, covariances, innovations, innovation_variances
    ))
    assert np.allclose(
        covariances, np.swapaxes(covariances, -1, -2), atol=1e-10
    )
    assert np.linalg.eigvalsh(covariances).min() >= -1e-10
    assert np.all(innovation_variances > 0)
    return True


### 2.1 Inspect and implement (25 points)

Plot truth and measurements and explain the separate random
streams. Complete the supplied Kalman-filter function using
`FILTER_M0`, `FILTER_P0`, and its stated update/predict ordering.
Run `validate_filter_outputs` on the returned arrays before
interpreting any result.


In [ ]:
# YOUR CODE HERE


### 2.2 Numerical validation (15 points)

Verify output shapes and finiteness, covariance symmetry to
`1e-10`, and positive semidefiniteness to numerical tolerance.
Check the first entries of the supplied discretization against
`F[0,0] = 0.99505374` and `F[0,1] = 0.04917539`.


In [ ]:
# YOUR CODE HERE


### 2.3 Nominal filtering performance (15 points)

With the correct process covariance, plot truth, observations,
filtered position/velocity, and 95% marginal bands. Report both
state RMSEs and the fraction of time points within each band.


In [ ]:
# YOUR CODE HERE


### 2.4 Process-noise misspecification (10 points)

Repeat with \(Q=0.05Q_{true}\) and \(Q=20Q_{true}\). Compare state
RMSE, average posterior SD, and standardized-innovation mean and
SD across all three cases.


In [ ]:
# YOUR CODE HERE


### 2.5 Interpretation (10 points)

Explain why the smoothest estimate need not be best, what the
innovation diagnostics reveal, and why the interval fraction from
one realized trajectory is not repeated-sampling coverage.


> **Response:** Replace this text with your work.
